### Data Ingestion

In [191]:
### document Structure

from langchain_core.documents import Document


In [192]:
doc = Document(
    page_content='this is the main text content im uisng to create RAG',
    metadata={
        "source":"example",
        "pages":1,
        "author":'Imran Khan',
        "date_created":'2026-03-22'
    }
)
doc

Document(metadata={'source': 'example', 'pages': 1, 'author': 'Imran Khan', 'date_created': '2026-03-22'}, page_content='this is the main text content im uisng to create RAG')

In [193]:
## Create a simple txt file
import os 
os.makedirs('../data/text_files',exist_ok = True)

In [194]:
sample_text = {
    "../data/text_files/rag_intro.txt":"""Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models by combining them with external knowledge sources. Instead of relying only on pre-trained data, RAG retrieves relevant information from a document store and uses it to generate more accurate and context-aware responses. Frameworks like LangChain make it easier to build RAG pipelines by integrating document loading, text splitting, embeddings, and vector databases. This approach is widely used in applications such as question answering systems, chatbots, and knowledge assistants."""
}
for filepath,content in sample_text.items():
    with open(filepath,'w',encoding = 'utf-8') as f:
        f.write(content)
print("Same text files created !")

Same text files created !


In [195]:
### TextLoader
from langchain_community.document_loaders import TextLoader
loader = TextLoader('../data/text_files/rag_intro.txt',encoding = 'utf-8')
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/rag_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models by combining them with external knowledge sources. Instead of relying only on pre-trained data, RAG retrieves relevant information from a document store and uses it to generate more accurate and context-aware responses. Frameworks like LangChain make it easier to build RAG pipelines by integrating document loading, text splitting, embeddings, and vector databases. This approach is widely used in applications such as question answering systems, chatbots, and knowledge assistants.')]


In [196]:
### Directory Loader incase you have got multiple files
from langchain_community.document_loaders import DirectoryLoader
#Load all txt files from directory
dir_loader = DirectoryLoader(
    '../data/text_files',glob='**/*.txt', ## Pattern to match files
    loader_cls= TextLoader, #Loader class to use
    loader_kwargs= {'encoding' : 'utf-8'},
    show_progress= False
)
files = print(dir_loader.load())

[Document(metadata={'source': '..\\data\\text_files\\rag_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models by combining them with external knowledge sources. Instead of relying only on pre-trained data, RAG retrieves relevant information from a document store and uses it to generate more accurate and context-aware responses. Frameworks like LangChain make it easier to build RAG pipelines by integrating document loading, text splitting, embeddings, and vector databases. This approach is widely used in applications such as question answering systems, chatbots, and knowledge assistants.')]


In [197]:

from langchain_community.document_loaders import PyMuPDFLoader

## PDF Loader

dir_loader = DirectoryLoader(
    '../data/pdf_files',glob='**/*.pdf', ## Pattern to match files
    loader_cls=PyMuPDFLoader, ##Loader class to use
    show_progress= False
)
pdf_files = dir_loader.load()
pdf_files


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-27T15:39:16+05:30', 'source': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'file_path': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Imran Khan', 'subject': '', 'keywords': '', 'moddate': '2026-02-27T15:39:16+05:30', 'trapped': '', 'modDate': "D:20260227153916+05'30'", 'creationDate': "D:20260227153916+05'30'", 'page': 0}, page_content='PHATAN MOHAMMAD IMRAN KHAN \nFrontend-Focused MERN Stack Developer | React & Node.js \nHyderabad, India \n  phatanmohammadimrankhan@gmail.com \n  +91-7013859797 \n \n \nPROFESSIONAL SUMMARY \nFrontend-focused MERN Stack Developer with 4+ years of experience building scalable, high-performance \nweb applications using ReactJS and integrating with Node.js/Express-based REST APIs. Strong expertise in \nReact, Redux Toolkit, TypeScript, API integration, authentication f

In [198]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=split_documents(pdf_files)
chunks

Split 3 documents into 6 chunks

Example chunk:
Content: PHATAN MOHAMMAD IMRAN KHAN 
Frontend-Focused MERN Stack Developer | React & Node.js 
Hyderabad, India 
  phatanmohammadimrankhan@gmail.com 
  +91-7013859797 
 
 
PROFESSIONAL SUMMARY 
Frontend-focused...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-27T15:39:16+05:30', 'source': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'file_path': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Imran Khan', 'subject': '', 'keywords': '', 'moddate': '2026-02-27T15:39:16+05:30', 'trapped': '', 'modDate': "D:20260227153916+05'30'", 'creationDate': "D:20260227153916+05'30'", 'page': 0}


[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-27T15:39:16+05:30', 'source': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'file_path': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Imran Khan', 'subject': '', 'keywords': '', 'moddate': '2026-02-27T15:39:16+05:30', 'trapped': '', 'modDate': "D:20260227153916+05'30'", 'creationDate': "D:20260227153916+05'30'", 'page': 0}, page_content='PHATAN MOHAMMAD IMRAN KHAN \nFrontend-Focused MERN Stack Developer | React & Node.js \nHyderabad, India \n  phatanmohammadimrankhan@gmail.com \n  +91-7013859797 \n \n \nPROFESSIONAL SUMMARY \nFrontend-focused MERN Stack Developer with 4+ years of experience building scalable, high-performance \nweb applications using ReactJS and integrating with Node.js/Express-based REST APIs. Strong expertise in \nReact, Redux Toolkit, TypeScript, API integration, authentication f

### Creating Embedding 

In [199]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [200]:
class EmbeddingManager:
    '''Handles Document Embedding Generation using SenetenceTransformer'''
    def __init__(self,model_name:str = 'all-MiniLm-L6-v2'):
        '''Initialize the embedding manager
            Args:
                model_name : HuggingFace model name for sentence embeddings
        '''
        self.model_name = model_name
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesion:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e :
            print(f"Error loading model {self.model.name} : {e}")
            raise
    
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        '''Generate embeddings for a list of text
            Args:
                texts: List of text strings to embed
            Returns:
                numpy arr of embeddings with shape (len(texts),embedding_dim)
        '''
        if not self.model:
            raise ValueError('Model not loaded')
        print(f"texts {texts}")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embedding with shape : {embeddings.shape}")
        return embeddings
##initialize embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLm-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4417.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLm-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimesion:384


### Creating VectoreStore DataBase

In [201]:
class VectorStore:
    '''Manages document embeddings in a ChromaDB vector store'''
    def __init__(self,collection_name:str = 'pdf_documents_cosine',persist_directory:str = '../data/vector_store'):
        '''Initialize the vector store
        Args:
            Collectoin_name : Name of the ChromaDB collection
            persist_director: Directory to persist the vector store
        '''
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        '''Initialize ChromaDB client and collection'''
        try:
                # Create persistent ChromaDB client
                os.makedirs(self.persist_directory,exist_ok= True)
                self.client = chromadb.PersistentClient(path=self.persist_directory)
                
                #Get or create collection
                # self.collection = self.client.get_or_create_collection (name= self.collection_name, metadata = {"description":"PDF Document embeddings for RAG"})
                self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine", "description": "PDF Document embeddings for RAG"}
                    )

                print(f'Vector store inititalized. Collection: {self.collection_name}')
                print(f'Existing documents in Collection: {self.collection.count()}')
                
        except Exception as e :
            print(f"Error initializing vector store : {e}")
            raise
    def add_documents(self,documents: List[Any],embeddings:np.ndarray):
        '''Add documents and their embeddings to the vector store
        Args:
            documents: List of LangChain documents
            embeddings: Correspondingn embeddings for the documents
        '''
        if len(documents) != len(embeddings):
            raise ValueError('Number of documents must match number of embeddings')
        
        print(f'Adding {len(documents)} documents to vector store ... ')
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #Prepare Meta Data
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            #Document content
            documents_text.append(doc.page_content)
            
            #Embedding
            embeddings_list.append(embedding.tolist())
        
        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings= embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection : {self.collection.count()}")
            
        except Exception as e :
            print(f"Error adding documents to vector store : {e}")
            raise
        
vectorstore  = VectorStore()
vectorstore

Vector store inititalized. Collection: pdf_documents_cosine
Existing documents in Collection: 18


In [202]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-27T15:39:16+05:30', 'source': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'file_path': '..\\data\\pdf_files\\FullStack-Resume-Imran.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Imran Khan', 'subject': '', 'keywords': '', 'moddate': '2026-02-27T15:39:16+05:30', 'trapped': '', 'modDate': "D:20260227153916+05'30'", 'creationDate': "D:20260227153916+05'30'", 'page': 0}, page_content='PHATAN MOHAMMAD IMRAN KHAN \nFrontend-Focused MERN Stack Developer | React & Node.js \nHyderabad, India \n  phatanmohammadimrankhan@gmail.com \n  +91-7013859797 \n \n \nPROFESSIONAL SUMMARY \nFrontend-focused MERN Stack Developer with 4+ years of experience building scalable, high-performance \nweb applications using ReactJS and integrating with Node.js/Express-based REST APIs. Strong expertise in \nReact, Redux Toolkit, TypeScript, API integration, authentication f

In [203]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
### Generate Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

###store in the vector db
vectorstore.add_documents(chunks,embeddings)


texts ['PHATAN MOHAMMAD IMRAN KHAN \nFrontend-Focused MERN Stack Developer | React & Node.js \nHyderabad, India \n  phatanmohammadimrankhan@gmail.com \n  +91-7013859797 \n \n \nPROFESSIONAL SUMMARY \nFrontend-focused MERN Stack Developer with 4+ years of experience building scalable, high-performance \nweb applications using ReactJS and integrating with Node.js/Express-based REST APIs. Strong expertise in \nReact, Redux Toolkit, TypeScript, API integration, authentication flows (JWT/OIDC), and performance \noptimisation. Well-versed in MERN architecture, middleware concepts, MongoDB schema basics, and \nsecure application design. Experienced in Agile environments, delivering enterprise-grade applications for \nglobal clients, including Aspen Publishing and Daimler. \n \n \nTECHNICAL SKILLS \nFrontend \nReactJS (Hooks), JavaScript (ES6+), TypeScript, HTML5, CSS3, Tailwind CSS, Material UI, Responsive Web \nDesign \nBackend (Exposure & Integration)', 'TECHNICAL SKILLS \nFrontend \nReactJ

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.42it/s]

Generated embedding with shape : (6, 384)
Adding 6 documents to vector store ... 
Successfully added 6 documents to vector store
Total documents in collection : 24


### Retriever Pipeline From VectorStore

In [204]:
class RAGRetriever:
    '''Handles query-based retrieval from the vector store'''
    def __init__(self,vector_store: VectorStore,embedding_manager = EmbeddingManager):
        '''
        Initializing the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        '''
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieval(self,query:str, top_k:int = 5, score_threshold :float = -1.0) -> List[Dict[str,Any]]:
        '''Retreive relevant documents for a query
        Args:
            query : The search query
            top_k : Number of top results to return
            score_threshold : Minimum similarity score threshold
        Returns:
            List of dictonaries containing retrieved documents and metadata
            
        '''
        print(f"Retreivig documents for query: {query}'")
        print(f"Top K : {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
                )
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,                                  
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank': i+1
                            
                            })
                print (f'Retrieved {len(retrieved_docs)} documents (after filtering)')
            else:
                print(f'No documents found')
            return retrieved_docs
        except Exception as e:
            print(f'Error during retrieval : {e}')
            return [] 
rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [205]:
rag_retriever

In [206]:
rag_retriever.retrieval('what is my total experience')

Retreivig documents for query: what is my total experience'
Top K : 5, Score threshold: -1.0
texts ['what is my total experience']
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 94.36it/s]

Generated embedding with shape : (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fafa1280_3',
  'content': '• \nCollaborated with backend developers to refine API contracts for updated multi-select evaluation \nlogic. \n• \nParticipated in API testing and validation using Postman and Swagger. \n• \nWorked in Agile sprints, resolving UAT and production issues. \n \nClient: Daimler \n \n \n \n \n \n \n \n \nJan – 2024 – Nov - 2024 \nProject: Volume Incentive Program (VIP) \nEnterprise-level incentive management platform. \nResponsibilities: \n• \nBuilt scalable React components and role-based dashboards driven by backend permission APIs. \n• \nIntegrated authentication using OIDC and IAM-based secure access. \n• \nConsumed and validated REST APIs developed in Node.js. \n• \nAssisted the backend team during debugging of authorisation and validation middleware issues. \n• \nOptimized performance using lazy loading and code splitting, reducing initial load time by 30–35%. \n• \nConfigured environment-specific variables securely using .env files, supported p

### Integrating Vector DB Context Pipeline with LLM O/P

In [207]:
### Simple RAG Pipeline with OpenAI
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True) 
key = os.getenv('OPENAI_API_KEY')

# Initialize the Open AI LLM 
llm = ChatOpenAI(
    model = 'gpt-4o-mini',
    max_completion_tokens=1024,temperature=0.1,api_key=key)

# Simple RAG Function : Retreive Context + Generate Response
def rag_simple(query,retreiver,llm,top_k=3):
    # retreive context from the retriever passed into the function
    result = retreiver.retrieval(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in result]) if result else ""
    if not context:
        return 'No relevant content found to answer the question'
    # Generate answer using OPEN AI LLM
    prompt = f"""use the following context to answer the question concisely
        Context : {context}
        Question: {query}
        Answer:
    """
    # response = llm.invoke([prompt.format(context = context , query = query)])
    response = llm.invoke(prompt)

    return response.content


In [215]:
answer = rag_simple('what is rag',rag_retriever,llm)
print(answer)

Retreivig documents for query: what is rag'
Top K : 3, Score threshold: -1.0
texts ['what is rag']
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 108.15it/s]

Generated embedding with shape : (1, 384)
Retrieved 3 documents (after filtering)


RAG typically stands for "Red, Amber, Green," a color-coding system used in project management to indicate the status of a project or task. Red signifies issues or problems, Amber indicates caution or potential risks, and Green means everything is on track.
